In [2]:
import os
import re
import json
import requests
from tqdm import tqdm
from glob import glob

DATA_DIR = '../static/data'
FILE_PATTERN = os.path.join(DATA_DIR, "graph_master_part100_*.json")
files = sorted(glob(FILE_PATTERN))
print(f"Found {len(files)} files.")

Found 194 files.


In [5]:
def fetch_confidence_scores(source, target):
    identifiers = f"{source}%0d{target}"  # URL-encoded list of IDs separated by carriage return
    results = [] # {}
    udf_results = { # placeholder for when interaction is not found
            "ncbiTaxonId": "N/A", 
            "score": 0.0, 
            "nscore": 0.0, 
            "fscore": 0.0, 
            "pscore": 0.0, 
            "ascore": 0.0, 
            "escore": 0.0, 
            "dscore": 0.0, 
            "tscore": 0.0
        }

    # finding edge between source and target
    found = False
    url = f"https://string-db.org/api/json/network?identifiers={identifiers}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        for edge in data:
            if (edge.get('preferredName_A') == source and edge.get('preferredName_B') == target) or \
                (edge.get('preferredName_A') == target and edge.get('preferredName_B') == source):
                
                score_types = ['ncbiTaxonId', 'score', 'nscore', 'fscore', 'pscore', 'ascore', 'escore', 'dscore', 'tscore']
                scores = {field: edge.get(field, 0.0) for field in score_types}
                score_vals = {k: v for k, v in scores.items() if k != 'ncbiTaxonId'}
                if any(value > 0 for value in score_vals.values()):
                    results.append({
                        **scores
                    })
                    found = True
                # break

    except Exception as e:
        print(f"Unexpected error: {e}")

    if not found:
        results.append(udf_results)
    return results

In [3]:
def extract_number(filename):
    match = re.search(r'(\d+)\.json$', filename)
    return int(match.group(1)) if match else -1

files = sorted(files, key=extract_number)
filelist = files[:5]
filelist

['../static/data/graph_master_part100_0.json',
 '../static/data/graph_master_part100_1.json',
 '../static/data/graph_master_part100_2.json',
 '../static/data/graph_master_part100_3.json',
 '../static/data/graph_master_part100_4.json']

In [ ]:
from gseapy import enrichr
from collections import defaultdict 

for file in tqdm(filelist):
    with open(file, "r") as f:
        data = json.load(f)

    # node analysis (gene set enrichment)
    nodes = data.get("nodes", [])
    gene_symbols = [str(node["id"]) for node in nodes]

    enr = enrichr(gene_list=gene_symbols,
                  gene_sets=["KEGG_2021_Human", "GO_Biological_Process_2021"],
                  outdir=None
                  )
    
    gene_to_terms = defaultdict(list)
    if enr.res2d is not None:
        for _, row in enr.res2d.iterrows():
            term = row["Term"]
            genes_in_term = row["Genes"].split(";")
            source = row["Gene_set"]
            for gene in genes_in_term:
                gene_to_terms[gene].append({"term": term, "source": source})

    # updating nodes with gene set info
    for node in nodes:
        gene_id = str(node["id"])
        node["gene_sets"] = gene_to_terms.get(gene_id, [])

    # link analysis (PPI confidence scores)
    links = data.get("links", [])
    new_links = []

    for link in links:
        src = str(link["source"])
        tgt = str(link["target"])
        id = str(link["id"])
        color = link.get("color") 
        penwidth = link.get("penwidth")

        scores = fetch_confidence_scores(src, tgt)
        for score_entry in scores:

            link_data = {
                "source": src,
                "target": tgt,
                "id": id,
                **score_entry
            }
            if color is not None:
                link_data["color"] = color
            if penwidth is not None:
                link_data["penwidth"] = penwidth
            
            new_links.append(link_data)

    data["links"] = new_links
    # saving to new folder
    OUTPUT_DIR = "../static/data/graph_master_scored"
    filename = os.path.basename(file)
    outname = os.path.join(OUTPUT_DIR, filename)
    with open(outname, "w") as f:
        json.dump(data, f, indent=2)


 20%|██        | 1/5 [00:04<00:19,  4.97s/it]

Unexpected error: 404 Client Error: Not Found for url: https://string-db.org/api/json/network?identifiers=zzzzccbbaaMBqqqqqqweweew333%0DMwewB33333


 40%|████      | 2/5 [04:05<07:11, 143.75s/it]